# ACE-Net Quick Smoke Test (5 Samples)
### Target Dataset: **TRACK_1**
### Purpose: Rapid pipeline & GPU validation in ~30 seconds before full run
### Output: `Google Drive > THESIS_MOTHERFILE > Baseline preprocessed > _TEST_RUNS > TRACK_1`

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Clone Repository & Checkout Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/training-and-preprocessing-jc-turnover
!git pull
!git log --oneline -1

## Step 3: Install Required Dependencies

In [ ]:
!pip install --prefer-binary -q openai-whisper facenet-pytorch
print('Dependencies installed successfully!')

## Step 4: Unzip Raw Dataset (`tracks_1_2_3_4.zip`) to Local Colab SSD

In [ ]:
import os, zipfile

DRIVE_ZIP = '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/tracks_1_2_3_4.zip'
LOCAL_RAW = '/content/data/raw/TRACK_1'

os.makedirs(LOCAL_RAW, exist_ok=True)
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f'Raw zip not found in Drive: {DRIVE_ZIP}')

print(f'Unzipping {DRIVE_ZIP} to local SSD ({LOCAL_RAW})...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z_file:
    z_file.extractall(LOCAL_RAW)
print('Unzip complete! Local files ready.')

## Step 5: Execute 5-Sample Smoke Test on `TRACK_1`

In [ ]:
!python scripts/preprocess/run_shard.py \
    --account 'test_runner@gmail.com' \
    --dataset 'TRACK_1' \
    --shard '0001' \
    --raw_dir '/content/data/raw/TRACK_1' \
    --drive_root '/content/drive/MyDrive/THESIS_MOTHERFILE/_TEST_RUNS' \
    --limit 5 \
    --device cuda

## Step 6: Verify Generated Feature Tensors (.npy & .jpg)

In [ ]:
import glob, numpy as np
from pathlib import Path

test_out = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/_TEST_RUNS/Baseline preprocessed/TRACK_1/shards/shard_0001')
mels = list(test_out.glob('audio/*.npy'))
texts = list(test_out.glob('text/*_input_ids.npy'))
vis = list(test_out.glob('visual/*'))

print('=' * 60)
print('SMOKE TEST AUDIT: TRACK_1')
print(f'Audio Melspecs Found : {len(mels)} / 5')
print(f'Text Token Sets Found: {len(texts)} / 5')
print(f'Visual Folders Found : {len(vis)} / 5')
if len(mels) > 0:
    sample_mel = np.load(mels[0])
    print(f'Sample Mel Shape (Target: [80, T]): {sample_mel.shape}')
print('=' * 60)
if len(mels) >= 1 and len(texts) >= 1:
    print('[SUCCESS] Pipeline verified working for TRACK_1!')
else:
    print('[NOTICE] Check logs above if files were skipped.')